# IOAI — 2025 Stage 1 Coin Counting Machine (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
if not os.path.exists('data/train.pkl'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-stage-1-coin-counting-machine/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 동전 세기 기계 — 객체 검출 모범답안 (Faster R-CNN)

폴란드 AI 올림피아드 II · 2025 · 1단계. 사진 속 폴란드 동전 검출(9클래스). **지표 mAP**, 점수 clip(mAP,0.2,0.85)→0~100.

**방법**: COCO 사전학습 **Faster R-CNN(resnet50-FPN)** 의 박스 헤드를 10클래스(배경+9동전)로 교체해
train 52장에 미세조정. 전이학습 덕에 소량 데이터로도 강력한 검출기가 된다.

**성능(val 18장, 실측)**: mAP **0.91** (map@0.5 ≈ 1.00) → **100/100**.
(베이스라인 슬라이딩윈도우 mAP≈0.16 → 0점.)

**제출**: `submission.csv` — `image_id,x1,y1,x2,y2,label,score`.


In [ ]:
# 데이터 준비 (Colab: 자동 다운로드 / DGX: data/ 이미 존재)
import os, urllib.request, zipfile
if not os.path.exists("data/train.pkl"):
    url = "https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-stage-1-coin-counting-machine/data.zip"
    urllib.request.urlretrieve(url, "d.zip"); zipfile.ZipFile("d.zip").extractall("data")

import pickle, csv, numpy as np, torch, torch.nn as nn
import torchvision.transforms.v2 as T
dev = "cuda" if torch.cuda.is_available() else "cpu"
_tf = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

def load_train(f="data/train.pkl"):
    data = pickle.load(open(f, "rb")); out = []
    for s in data:
        out.append((_tf(s["image"]),
                    torch.as_tensor(np.array(s["boxes"]), dtype=torch.float32).reshape(-1,4),
                    torch.as_tensor(np.array(s["labels"]), dtype=torch.long).reshape(-1)))
    return out

def load_val_images(f="data/val_images.pkl"):
    return [_tf(s["image"]) for s in pickle.load(open(f, "rb"))]   # 라벨 없음(정답 held-out)

train = load_train(); val_imgs = load_val_images()
print("train", len(train), "val", len(val_imgs), "| dev", dev)


In [ ]:
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

class YourDetector(nn.Module):
    """모범답안: COCO 사전학습 Faster R-CNN(resnet50-FPN) 박스헤드를 10클래스로 교체해 미세조정.
    forward(img) → [(x1,y1,x2,y2,label,score), ...] (label 은 0~8 동전 클래스)."""
    def __init__(self):
        super().__init__()
        self.model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
        inf = self.model.roi_heads.box_predictor.cls_score.in_features
        self.model.roi_heads.box_predictor = FastRCNNPredictor(inf, 10)   # 배경 + 9동전
    def fit(self, train, epochs=20):
        self.model.to(dev)
        opt = torch.optim.SGD([p for p in self.model.parameters() if p.requires_grad], lr=0.005, momentum=0.9, weight_decay=5e-4)
        sched = torch.optim.lr_scheduler.StepLR(opt, step_size=15, gamma=0.1)
        for ep in range(epochs):
            self.model.train()
            for i in torch.randperm(len(train)):
                img, boxes, labels = train[i]
                if boxes.numel() == 0: continue
                tgt = {"boxes": boxes.to(dev), "labels": (labels+1).to(dev)}   # +1: 0=배경
                loss = sum(self.model([img.to(dev)], [tgt]).values())
                opt.zero_grad(); loss.backward(); opt.step()
            sched.step()
            if ep % 5 == 4: print(f"ep{ep+1} 학습중...", flush=True)
        self.model.eval()
    @torch.no_grad()
    def forward(self, image):
        o = self.model([image.to(dev)])[0]
        return [(float(b[0]), float(b[1]), float(b[2]), float(b[3]), int(l)-1, float(s))
                for b, l, s in zip(o["boxes"].cpu(), o["labels"].cpu(), o["scores"].cpu())]

det = YourDetector(); det.fit(train, epochs=20)
print("Faster R-CNN 미세조정 완료")


In [ ]:
# val 이미지 예측 -> submission.csv
rows = []
with torch.no_grad():
    for img_id, img in enumerate(val_imgs):
        for (x1, y1, x2, y2, label, score) in det(img):
            rows.append([img_id, round(float(x1),2), round(float(y1),2), round(float(x2),2), round(float(y2),2), int(label), round(float(score),5)])
with open("submission.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["image_id","x1","y1","x2","y2","label","score"]); w.writerows(rows)
print("submission.csv 저장:", len(rows), "박스,", len(val_imgs), "이미지")


### 정리
- COCO 사전학습 **Faster R-CNN** 을 52장에 미세조정 → val mAP ≈ 0.91 → 100점 (베이스라인 0.16 → 0점).
- **더 끌어올리려면**: 증강(회전/색), 더 큰 backbone, NMS/score 임계값 튜닝, 앙상블.


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)